# Analyze

One saved run, one block per analysis. The first cell is the only setup: name the topic and the run, say whether anything is written, and whether figures use the topic's `plot.json` (`STYLED`) or the default palette. With `SAVE = False` every block only shows its table and figure; with `True` tables go to `results/<topic>/analysis/` and figures to `results/<topic>/plots/`. `python analyze.py <topic> <run>` runs the same blocks in one go. `PREDICTORS` names the columns the effects block tests (a list); `None` takes the dataset's grouping columns and its claims column.

In [ ]:
%matplotlib inline
from panel import paths
from panel.analysis import Output, answers, area, compass, correlation, country, describe, distance, effects, find_stem, groups, load_run, spread
from panel.run import ResultStore, resolve
from panel.run.check import check, report

TOPIC = "election_2021"
STEM = "sod_default_cs_default_Anthropic_claude-haiku-5-5"
SAVE = False
PREDICTORS = None
STYLED = True

STORE = ResultStore(paths.RESULTS_DIR, TOPIC)
RUN = load_run(STORE, find_stem(STORE, STEM), styled=STYLED)
OUT = Output(STORE, RUN.stem, save=SAVE)
print(describe(RUN))

## What the run saved

Is the run complete and each answer consistent - from the saved files alone, compared with nothing. It also counts the answers that give going to vote a probability of zero, those whose options sum to zero (no preference can be read from them), and how far the two overlap.

In [ ]:
print(report(check(resolve(RUN.spec), STORE), len(RUN.options)))

## What the answers say

The mean of every column as saved, and for each option the share of answers in which it is the highest. A table only.

In [ ]:
table, figures = answers(RUN, OUT)
display(table.round(3), *figures)

## The run's subjects

In [ ]:
RUN.subjects.join(RUN.normalized).head()

## Country level

Predicted, claimed and actual over all the run's subjects. Claimed is counted from the same subjects' own answers; whiskers are bootstrap intervals over subjects.

In [ ]:
table, figures = country(RUN, OUT)
display(table.round(3), *figures)

## Distances between sources

Predicted from actual, predicted from claimed, claimed from actual, over the options: MAE, RMSE, correlation, Jensen-Shannon and KL divergence, and the turnout difference. One resampling of subjects serves all pairs.

In [ ]:
table, figures = distance(RUN, OUT)
display(table.round(4), *figures)

## By area

MAE and RMSE between each pair of sources in every area the topic has a result for, the whole first, then areas by how many of the run's subjects live there. One figure per metric; its three panels share one scale. `results/<topic>/analysis/area_shares_<stem>.csv` holds the distributions behind it.

In [ ]:
table, figures = area(RUN, OUT)
display(table.round(3), *figures)

## Spread of answers per subject

How each column is distributed over subjects, and how far each subject's answer sits from the mean answer. Repeats are averaged per subject first; subjects with no preference are left out. `spread_distances_<stem>.csv` holds the distances per subject.

In [ ]:
table, figures = spread(RUN, OUT)
display(table.round(3), *figures)

## Correlation and covariance between options

Over subjects, on the same rows as the spread. The shares of a subject sum to 1, so the options are negatively correlated by construction; the figure's note says what no relation looks like. `covariance_<stem>.csv` holds the second matrix.

In [ ]:
table, figures = correlation(RUN, OUT)
display(table.round(2), *figures)

## By group

The predicted distribution at each level of every grouping column the dataset names, one figure per column, levels labelled with their subject count. Level order, age bands and columns where a subject may hold several levels (employment) come from `levels` in the dataset's `dataset.json`. `groups_claimed_<stem>.csv` holds what the same subjects claimed, `groups_spread_<stem>.csv` the standard deviation of each column within a level.

In [ ]:
table, figures = groups(RUN, OUT)
display(table.round(3), *figures)

## What the answers follow

Each grouping column tested against the per-subject option shares. MANOVA takes all columns together and gives what one accounts for given the others (partial eta squared, with F and p); PERMANOVA takes a column at a time, on Hellinger distances, and gives what it accounts for alone (R squared, with a permutation p).

In [ ]:
table, figures = effects(RUN, OUT, PREDICTORS)
display(table.round(3), *figures)

## The model's map of the options against the experts'

Two options are close for the model when it raises and lowers them for the same subjects. Those distances are drawn as a 2-D map (MDS) and laid onto the plane experts place the options in (`positions.json` of the topic: CHES 2019, economic left-right by GAL-TAN). Filled dots are the model's map, hollow ones the experts; the p-value is the share of relabellings of the options that fit at least as well. `compass_fit_<stem>.csv` holds the fit.

In [ ]:
table, figures = compass(RUN, OUT)
display(table.round(2), *figures)